# Weighted A*: proof and reproducible comparisons
This notebook checks the implemented `route_model.weighted_astar` against an independent Dijkstra search. It is evidence about this implementation and dataset, not a substitute for the lower-bound argument below. Run from the repository checkout with Python dependencies in `src/backend/osm/requirements-dev.txt` installed.


## Lower-bound argument
For every traversable edge e, let its positive travel time be t_e, its nonnegative exposure index be I_e, and its endpoints have great-circle distance d_e. Let k = min(t_e / d_e) over edges with d_e > 0 (or zero if none), and I_min = min(I_e) over traversable edges. For a node n and destination g, set t_min(n) = k × great-circle-distance(n, g). By the triangle inequality and the definition of k, every remaining path takes at least t_min(n). Its exposure sum is at least I_min times its actual time, hence at least I_min × t_min(n).
With nonnegative α and β and positive normalization denominators D_T and D_E, h(n) = α t_min(n)/D_T + β I_min t_min(n)/D_E is no greater than the cost of any remaining path. D_T = T_ref if positive, otherwise 1 second; D_E = E_ref if positive, otherwise 1 index-second. This preserves the existing zero-reference convention but is not relative normalization when a reference is zero.
The same argument gives consistency: the change in straight-line time bound across an edge is at most k × d_e ≤ t_e, and the heuristic's per-second factor is at most the edge's weighted cost per second. Zero-length geographic separations impose no restriction on k. Directed edges, imputed values, custom edge times and disconnected components are included in the minimum if traversable; invalid or missing exposure values are rejected. Nonnegative costs and a valid finite graph are required. Floating-point calculations and their implementation still require empirical verification.


In [ ]:
from pathlib import Path
import sys, random, heapq, time, math
from dataclasses import replace
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src/backend/route_model').is_dir())
sys.path.insert(0, str(root / 'src/backend'))
from planner.models import Preference
from route_model.costs import edge_exposure, weighted_cost
from route_model.weighted_astar import weighted_astar
from solver.tests.biggrid import grid_input


## Comparison
Both weighted searches use the same graph validation, adjacency, edge costs, reference values and search implementation. Dijkstra uses a zero heuristic; A* uses the documented lower bound. We time shared preparation, A*-specific heuristic setup, and each search separately. The retained research weighted A* result is independently checked against both; the default mock comparator uses weighted Dijkstra. Synthetic grid runs are illustrative, not Railway performance.


In [ ]:
sys.path.insert(0, str(root / 'notebooks'))
from weighted_astar_live import _run_searches
rng = random.Random(42)
si = grid_input(25, reachable=True)
si.edges = [replace(e, travel_time_s=e.travel_time_s * rng.uniform(.8, 1.2), exposure_index=rng.uniform(.5, 3.0)) for e in si.edges]
for alpha in (0.0, .25, .5, .75, 1.0):
    si.preference = Preference(alpha, 1 - alpha)
    print(alpha, _run_searches(si, repeats=3))


The test suite additionally checks one-way edges, zero and missing exposure, zero references, cancellation, unreachable nodes and hand-calculated preference trade-offs. A positive global minimum can be weak; if the minimum index is zero, the exposure contribution to the heuristic is zero. A* may expand fewer nodes without taking less wall time because it also computes geographic distances.


## Optional live database comparison
The application planner reads the active graph and latest complete pollution window through its read-only pool. Set DATABASE_URL in `src/backend/.env` or the environment; never put credentials in notebook output. A read-only role is recommended because session read-only is not an access-control boundary. Disabled by default.
For each scenario and preference, `reference_ms` measures the common fastest-route reference search once; `shared_prepare_ms` is median validation, adjacency and weighted-cost preparation used by both; `astar_heuristic_setup_ms` is A*-specific lower-bound preparation; and the two search times use the exact same prepared graph and search code, with only the heuristic differing. Search timings are warmed, alternate order and report the median of three runs. The retained research A* implementation is separately checked for equal cost but is not used as the search-only timer. The former separate weighted provider has been removed; the mock is the default deterministic weighted Dijkstra comparator. `planner_ms` includes database reads and preparation. Runs are local, not full API or Railway server timings.


In [ ]:
RUN_LIVE_DB = False  # Explicit opt-in: opens a read-only database session.
if RUN_LIVE_DB:
    sys.path.insert(0, str(root / 'notebooks'))
    from weighted_astar_live import benchmark_live
    live_results = await benchmark_live(repeats=3)
    for row in live_results:
        print(row)
else:
    print('Live database benchmark disabled; set RUN_LIVE_DB = True to run.')


### Historical provider comparison (30 October 2026)
One local Docker run used graph `jakarta-20261005` and the 2026-10-28 13:00 UTC pollution window (flagged stale), with no imputed edges in these inputs. Provider times below are medians of three runs for α = β = 0.5, excluding planner/database time. These are historical measurements, not live results.
| Scenario | Nodes / edges | Weighted A* provider | Mock Dijkstra provider |
| --- | ---: | ---: | ---: |
| Central short, walk | 3,444 / 4,282 | 124.32 ms | 29.55 ms |
| Central long, walk | 17,702 / 21,966 | 678.27 ms | 291.00 ms |
| Central long, bike | 15,394 / 18,583 | 566.39 ms | 177.16 ms |
In all nine scenario/preference combinations weighted A* and Dijkstra had equal optimal costs and the two providers' selected reference totals matched. Weighted A* provider medians exceeded mock provider medians in all nine. The real planner reads took roughly 3.8–7.3 s in this run over the public connection; their timeout limits were raised for the exploratory benchmark, and they are not production API latency measurements. The weighted provider and mock have different setup/reference implementations, so these timings do not isolate the algorithm.
These are historical provider timings from before the separate weighted provider was removed, not the fair search-only comparison above. The earlier timed search comparison used unequal preparation and is superseded.


### Historical expansion and route comparison (30 October 2026)
Using the same graph and pollution window, this rerun counted node expansions for both searches. The shortest-path algorithms returned identical edge paths, travel times, exposure totals and weighted costs in all nine cases. Ties can produce different paths on other graphs.
| Scenario | α | A* expansions | Dijkstra expansions |
| --- | ---: | ---: | ---: |
| Short walk | 0 | 1,740 | 2,611 |
| Short walk | 0.5 | 1,267 | 2,643 |
| Short walk | 1 | 705 | 2,684 |
| Longer walk | 0 | 9,850 | 15,383 |
| Longer walk | 0.5 | 7,237 | 15,823 |
| Longer walk | 1 | 3,685 | 15,705 |
| Longer bike | 0 | 8,015 | 12,279 |
| Longer bike | 0.5 | 6,859 | 12,221 |
| Longer bike | 1 | 5,293 | 12,646 |
The longer walking route showed a trade-off across preferences: α = 0 gave 4,299.838 s and 24,976.3399 index·s; α = 0.5 gave 4,102.572 s and 26,006.5809 index·s; α = 1 gave 4,057.349 s and 27,731.6086 index·s. Both algorithms selected each same route. Counts are valid expansions (non-stale priority queue entries), not necessarily unique nodes; Dijkstra uses the same adjacency and edge ordering as the backend search.


### Shared-preparation benchmark (30 October 2026)
Graph `jakarta-20261005`, pollution window 28 October 13:00 UTC (stale), no imputed edges. Local Docker; three search repetitions after warm-up. Both search algorithms use the same weighted costs, adjacency, code and reference values. The weighted A* implementation was checked against their minimum cost; it was the provider implementation at the time of this historical run. Times below are milliseconds.
| Scenario | α | Shared preparation | A* heuristic setup | A* search | Dijkstra search | A* / Dijkstra expansions |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| Short walk | 0 | 11.78 | 5.55 | 3.95 | 2.59 | 1,740 / 2,611 |
| Short walk | 0.5 | 11.40 | 4.97 | 2.66 | 2.41 | 1,267 / 2,643 |
| Short walk | 1 | 14.98 | 6.37 | 1.51 | 2.60 | 705 / 2,684 |
| Longer walk | 0 | 94.66 | 33.16 | 29.30 | 21.50 | 9,850 / 15,383 |
| Longer walk | 0.5 | 100.56 | 30.91 | 23.38 | 25.62 | 7,237 / 15,823 |
| Longer walk | 1 | 70.61 | 32.07 | 10.23 | 25.24 | 3,685 / 15,705 |
| Longer bike | 0 | 77.05 | 25.70 | 20.45 | 14.61 | 8,015 / 12,279 |
| Longer bike | 0.5 | 90.60 | 27.19 | 22.62 | 19.29 | 6,859 / 12,221 |
| Longer bike | 1 | 89.01 | 30.47 | 17.23 | 19.88 | 5,293 / 12,646 |
All paths and costs matched for these cases. A* search alone was faster in four of nine cases, but A*-specific heuristic setup exceeded its search-time savings in every case. Shared preparation and reference search were measured separately; adding individual medians gives an approximation, not the median of a combined run. Preparation and search-only comparisons use benchmark code, not a claim about current provider latency. The graph-dependent heuristic factor could potentially be cached with appropriate invalidation, but that was not measured here.
